# 03 - Uncertainty Estimation (GDA Epistemic)

Fits the Gaussian Discriminant Analysis (GDA) model, computes per-point epistemic uncertainty for **train and test separately**, and saves the uncertainty values as columns in augmented split files

In [ ]:
import sys
from pathlib import Path

#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
#sys.path.insert(0, str(Path.cwd().parent if (Path.cwd() / "notebooks").exists() else Path.cwd()))
current_notebook_dir = Path.cwd()

if current_notebook_dir.name == "notebooks":
    project_root = current_notebook_dir.parent
else:
    project_root = current_notebook_dir

if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from src.utils import set_global_seed, load_config, get_logger
from data.paths import get_dataset_config, get_split_file
from src.classes import get_class_registry
from src.features import DEFAULT_FEATURE_NAMES
from src.uncertainty import GDAUncertaintyModel, select_gda_feature_columns

logger = get_logger("03_uncertainty_estimation")

In [ ]:
DATASET = "ahn3"

config = load_config("default")
set_global_seed(config["experiment"]["seed"])
dataset_cfg = get_dataset_config(DATASET)
class_registry = get_class_registry(DATASET)

## 1. Load canonical train/test splits

In [ ]:
train_df = pd.read_csv(get_split_file(DATASET, "train"))
test_df  = pd.read_csv(get_split_file(DATASET, "test"))

print(f"Train: {len(train_df):,} rows | Test: {len(test_df):,} rows")
print(f"Train files: {sorted(train_df['__source_file__'].unique())}")
print(f"Test  files: {sorted(test_df['__source_file__'].unique())}")

## 2. Select GDA feature columns
- x,y,z are excluded from the uncertainty modeling part

In [ ]:
feature_columns = ["x", "y", "z"] + DEFAULT_FEATURE_NAMES
gda_columns = select_gda_feature_columns(feature_columns, config)

print(f"Total feature columns : {len(feature_columns)}")
print(f"GDA input columns     : {len(gda_columns)} (xyz excluded: {not config['uncertainty'].get('include_xyz_in_gda', False)})")
print(f"GDA columns: {gda_columns}")

## 3. Fit GDA
- Estimates per-class mean vectors μ_c and covariance matrices Σ_c from the training data

In [ ]:
X_train = train_df[gda_columns].values
y_train = train_df["class_idx"].values

gda = GDAUncertaintyModel(
    feature_columns=gda_columns,
    regularization=config["uncertainty"]["covariance_regularization"],
)
gda.fit(X_train, y_train)

print(f"GDA fitted on {len(X_train):,} train points across {len(gda.mu)} classes.")

## 4. Compute epistemic uncertainty - train and test separately

**Train**: scored against a distribution fit on itself

**Test**: scored against a similar distribution (train) comparable uncertainty.

In [ ]:
X_test = test_df[gda_columns].values

u_train_raw = gda.epistemic_uncertainty(X_train)
u_test_raw  = gda.epistemic_uncertainty(X_test)

u_min, u_max = u_train_raw.min(), u_train_raw.max()
u_train = GDAUncertaintyModel.normalize(u_train_raw, u_min=u_min, u_max=u_max)
u_test  = GDAUncertaintyModel.normalize(u_test_raw,  u_min=u_min, u_max=u_max)

print(f"Train uncertainty - min: {u_train.min():.4f}, max: {u_train.max():.4f}, "
      f"mean: {u_train.mean():.4f}")
print(f"Test  uncertainty - min: {u_test.min():.4f},  max: {u_test.max():.4f},  "
      f"mean: {u_test.mean():.4f}")

## 5. Visualise uncertainty distribution

Per-class uncertainty histograms and the overall train vs. test comparison. High uncertainty concentrated in minority classes with few is expected and desirable. This means the GDA's uncertainty signal is geometrically meaningful, not flat noise

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

# Overall train vs test
axes[0].hist(u_train, bins=50, alpha=0.6, label="Train", color="steelblue", density=True)
axes[0].hist(u_test,  bins=50, alpha=0.6, label="Test",  color="tomato",    density=True)
axes[0].set_xlabel("Normalised epistemic uncertainty")
axes[0].set_ylabel("Density")
axes[0].set_title("Train vs Test uncertainty distribution")
axes[0].legend()

class_names = class_registry.class_names_in_order()
for idx, name in enumerate(class_names):
    mask = test_df["class_idx"].values == idx
    if mask.sum() > 0:
        axes[1].hist(u_test[mask], bins=30, alpha=0.5, label=name, density=True)
axes[1].set_xlabel("Normalised epistemic uncertainty")
axes[1].set_ylabel("Density")
axes[1].set_title("Per-class uncertainty distribution (test split)")
axes[1].legend(fontsize=7, loc="upper right")

plt.tight_layout()
plt.savefig(str(Path(get_split_file(DATASET, "train")).parent / "uncertainty_distributions.png"),
            dpi=120, bbox_inches="tight")
plt.show()

## 6. Save the splits
- Adds `__u_raw__` and `__u_norm__` columns to both splits and writes them back to the canonical split paths

In [ ]:
train_df["__u_raw__"]  = u_train_raw
train_df["__u_norm__"] = u_train
test_df["__u_raw__"]   = u_test_raw
test_df["__u_norm__"]  = u_test

train_out = get_split_file(DATASET, "train")
test_out  = get_split_file(DATASET, "test")

train_df.to_csv(train_out, index=False)
test_df.to_csv(test_out,   index=False)

print(f"Saved train split with uncertainty : {train_out}")
print(f"Saved test  split with uncertainty : {test_out}")
print(f"Columns: {[c for c in train_df.columns if c.startswith('__')]}")

## 7. Per-class uncertainty summary (test)

In [ ]:
rows = []
for idx, name in enumerate(class_names):
    mask = test_df["class_idx"].values == idx
    if mask.sum() > 0:
        rows.append({
            "class_idx": idx,
            "class_name": name,
            "n_points": int(mask.sum()),
            "u_mean": round(float(u_test[mask].mean()), 4),
            "u_median": round(float(np.median(u_test[mask])), 4),
            "u_std": round(float(u_test[mask].std()), 4),
        })

summary = pd.DataFrame(rows).sort_values("u_mean", ascending=False)
print(summary.to_string(index=False))
